# 02 · RFT generation + baseline eval (Qwen3.6-35B-A3B, vLLM)

**RFT (rejection sampling fine-tuning):** the model writes several solutions to verifiable problems, graders keep the
correct ones, and the **shortest correct solution** for each problem becomes training data.
Training happens later in `04_sft.ipynb`.

**This notebook does two things:**
- **A) Baseline eval:** runs the sets under `eval/tasks` in the same session. What would take hours locally takes minutes here.
- **B) RFT generation:** a pool of math (DeepScaleR), physics (WebInstruct-verified), Python (KodCode, pytest) and tool-call (xLAM) problems.

**Runtime and cost:**
- Runtime: **A100 80GB** (≈6.8 CU/hour). The model takes ~35 GB in FP8 and runs on vLLM.
- `MAX_GEN_HOURS` caps the time budget. 5 hours ≈ 34 CU.
- If the session drops, rerun the notebook from the top. Generation resumes from the file on Drive.

**Before you start:**
1. Copy this repo's `eval/` folder to Drive as `MyDrive/colab-ai/eval/` (`run_eval.py`, `fetch_benchmarks.py`, `tasks/personal.jsonl`).
2. Add `HF_TOKEN` in Colab's **Secrets** (🔑) panel.
3. Accept the terms of these two gated datasets on Hugging Face:
   - [GPQA](https://huggingface.co/datasets/Idavidrein/gpqa)
   - [xLAM](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k)

   If a dataset is not accessible, that set is skipped and the notebook continues.

In [ ]:
# "pip's dependency resolver ..." warnings (cudf, cuml, numba, google-adk) are harmless: those packages are not used
!pip install -q -U vllm pytest

In [ ]:
import vllm, torch
print(vllm.__version__, torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Settings

In [ ]:
MODEL = "Qwen/Qwen3.6-35B-A3B-FP8"
MAX_MODEL_LEN = 20480
CU_PER_HOUR = 6.77                 # A100 80GB (as shown in Colab's Resources panel)

DRIVE_ROOT = "/content/drive/MyDrive/colab-ai"
EVAL_DIR = f"{DRIVE_ROOT}/eval"
RFT_DIR = f"{DRIVE_ROOT}/rft/round1"
SEED = 3407

MAX_NUM_SEQS = 512                 # concurrent requests (batch generation throughput)

# A) Baseline eval (set to False once done; the post-training comparison happens in 04)
RUN_EVAL = False
EVAL_LABEL = "base-colab-fp8"
EVAL_SETS = ["personal", "math500", "aime2025", "gpqa_physics"]
EVAL_SAMPLES = 4                   # 4 samples per problem: mean accuracy is less noisy than a single attempt
EVAL_TEMPERATURE = 0.6             # same as the local run_eval.py
EVAL_MAX_TOKENS = 16384

# B) RFT generation
# The baseline eval ran at ~2.1k tok/s: this pool ≈ 40M tokens ≈ 5 hours ≈ 34 CU
POOL = {"math": 500, "phys": 400, "code": 400, "tool": 300}
N_SAMPLES = 4
GEN_TEMPERATURE = 0.8
GEN_MAX_TOKENS = 12288             # longer solutions are dropped: the point is to teach shorter reasoning
CHUNK = 256
MAX_GEN_HOURS = 5.0
EASY_KEEP = 0.3                    # fraction of easy problems (solved 4/4) to keep

In [ ]:
import os, sys, json, glob, random, time, hashlib, collections, importlib
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
login(token=userdata.get("HF_TOKEN"))
os.makedirs(RFT_DIR, exist_ok=True)

if not os.path.exists(f"{EVAL_DIR}/run_eval.py"):
    found = glob.glob("/content/drive/MyDrive/**/run_eval.py", recursive=True)
    raise FileNotFoundError(f"{EVAL_DIR}/run_eval.py not found. Upload the repo's eval/ folder there "
                            f"or fix DRIVE_ROOT. run_eval.py files found on Drive: {found}")
sys.path.insert(0, EVAL_DIR)
sys.path_importer_cache.pop(EVAL_DIR, None)   # drop the stale "no such path" entry if the folder was uploaded later
importlib.invalidate_caches()
import run_eval as R          # the same graders as the local eval
import fetch_benchmarks

for name, fn in [("math500", fetch_benchmarks.math500), ("aime2025", fetch_benchmarks.aime2025),
                 ("gpqa_physics", fetch_benchmarks.gpqa_physics)]:
    if not os.path.exists(f"{EVAL_DIR}/tasks/{name}.jsonl"):
        try:
            fetch_benchmarks.write(name, fn())
        except Exception as e:
            print(f"{name}: DOWNLOAD FAILED -> {type(e).__name__}: {e}")

!nvidia-smi --query-gpu=name,memory.total --format=csv

## Model (vLLM)

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model = MODEL,
    max_model_len = MAX_MODEL_LEN,
    gpu_memory_utilization = 0.92,
    max_num_seqs = MAX_NUM_SEQS,
    limit_mm_per_prompt = {"image": 0, "video": 0},   # text only: no memory reserved for the vision encoder
    enable_prefix_caching = True,
    seed = SEED,
)
tok = llm.get_tokenizer()

In [ ]:
def to_prompt(task):
    return tok.apply_chat_template(R.build_messages(task), tools=task.get("tools"),
                                   tokenize=False, add_generation_prompt=True)


def generate(tasks, n, temperature, max_tokens):
    params = SamplingParams(n=n, temperature=temperature, top_p=0.95, top_k=20,
                            max_tokens=max_tokens, seed=SEED)
    outs = llm.generate([to_prompt(t) for t in tasks], params)
    samples = [[{"text": o.text, "tokens": len(o.token_ids), "finish": o.finish_reason} for o in out.outputs]
               for out in outs]
    # Grading runs in parallel: python/pytest tests run in subprocesses
    flat = [(t, s) for t, ss in zip(tasks, samples) for s in ss]
    with ThreadPoolExecutor(max_workers=(os.cpu_count() or 4) * 2) as ex:
        results = list(ex.map(lambda ts: R.grade(ts[0], {"content": ts[1]["text"]}), flat))
    for (t, s), (ok, detail, _) in zip(flat, results):
        s["ok"] = bool(ok) and s["finish"] == "stop"   # a truncated solution never counts as correct
        s["detail"] = str(detail)[:300]
    return samples


# Smoke test: are the template and the output format right?
print(to_prompt({"type": "math", "prompt": "What is 2+2?"}))
print(generate([{"id": "t", "type": "math", "prompt": "What is 2+2?", "answer": "4"}], 1, 0.6, 2048)[0][0])

## A) Baseline eval

- Results are appended to `eval/results/summary.csv` on Drive, in the same format as the local file.
- **Note:** this eval uses the FP8 model, so it can differ slightly from the Q4 quant used locally.
- **Fair comparison:** the fine-tuned model is evaluated the same way on Colab, so both runs share the same conditions.

In [ ]:
if RUN_EVAL:
    eval_files = [f"{EVAL_DIR}/tasks/{n}.jsonl" for n in EVAL_SETS if os.path.exists(f"{EVAL_DIR}/tasks/{n}.jsonl")]
    eval_tasks = R.load_tasks(eval_files, 0)
    print(f"{len(eval_tasks)} tasks × {EVAL_SAMPLES} samples")

    t0 = time.time()
    samples = generate(eval_tasks, EVAL_SAMPLES, EVAL_TEMPERATURE, EVAL_MAX_TOKENS)
    elapsed = time.time() - t0

    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    results_dir = f"{EVAL_DIR}/results"
    os.makedirs(results_dir, exist_ok=True)
    stats = {}
    with open(f"{results_dir}/{EVAL_LABEL}_{stamp}.jsonl", "w", encoding="utf-8") as f:
        for t, ss in zip(eval_tasks, samples):
            st = stats.setdefault(t["_source"], {"n": 0, "correct": 0, "tokens": 0, "seconds": 0.0})
            for s in ss:
                st["n"] += 1
                st["correct"] += int(s["ok"])
                st["tokens"] += s["tokens"]
                f.write(json.dumps({"id": t["id"], "source": t["_source"], "type": t["type"], "ok": s["ok"],
                                    "detail": s["detail"], "tokens": s["tokens"], "finish": s["finish"],
                                    "response": R.strip_thinking(s["text"])}, ensure_ascii=False) + "\n")
    total = sum(st["n"] for st in stats.values())
    for st in stats.values():
        st["seconds"] = elapsed * st["n"] / total   # batch generation: time is split proportionally across sources
    R.write_summary(results_dir, stamp, EVAL_LABEL, MODEL, stats)
    print(f"Time: {elapsed / 60:.1f} min ≈ {elapsed / 3600 * CU_PER_HOUR:.1f} CU")

## B) RFT pool

Every source has verifiable answers. Problems that exactly match an eval set are removed from the pool.

| Source | Dataset | Grader |
|---|---|---|
| math | `agentica-org/DeepScaleR-Preview-Dataset` (AIME ≤2023, AMC, Omni-MATH) | `\boxed{}` match |
| phys | `TIGER-Lab/WebInstruct-verified`, Physics, numeric answers | 2% relative tolerance |
| code | `KodCode/KodCode-V1`, instruct, medium/hard | pytest (license: CC BY-NC 4.0) |
| tool | `Salesforce/xlam-function-calling-60k` (gated) | all calls + arguments |

In [ ]:
from datasets import load_dataset

eval_prompts = set()
for p in glob.glob(f"{EVAL_DIR}/tasks/*.jsonl"):
    eval_prompts |= {json.loads(l)["prompt"].strip() for l in open(p, encoding="utf-8") if l.strip()}


def uid(prefix, text):
    return f"{prefix}-{hashlib.md5(text.encode()).hexdigest()[:12]}"


def pool_math(n):
    ds = load_dataset("agentica-org/DeepScaleR-Preview-Dataset", split="train").shuffle(seed=SEED)
    rows = (r for r in ds if r["problem"].strip() not in eval_prompts)
    return [{"id": uid("math", r["problem"]), "type": "math", "prompt": r["problem"], "answer": r["answer"]}
            for _, r in zip(range(n), rows)]


def pool_phys(n):
    ds = load_dataset("TIGER-Lab/WebInstruct-verified", split="train")
    ds = ds.filter(lambda r: r["category"] == "Physics"
                   and r["answer_type"] in ("Float", "Integer", "Percentage")
                   and r["difficulty"] in ("Senior High School", "University", "PhD"))
    ds = ds.shuffle(seed=SEED).select(range(min(n, len(ds))))
    return [{"id": uid("phys", r["question"]), "type": "numeric", "prompt": r["question"], "answer": r["answer"]}
            for r in ds]


def pool_code(n):
    stream = load_dataset("KodCode/KodCode-V1", split="train", streaming=True).shuffle(seed=SEED, buffer_size=20_000)
    out = []
    for r in stream:
        if (r["style"] != "instruct" or r["gpt_difficulty"] not in ("medium", "hard")
                or not r["test_info"] or "from solution import" not in r["test"]):
            continue
        sigs = "\n".join(f"    {ti['function_declaration']}" for ti in r["test_info"])
        prompt = f"{r['question']}\n\nImplement the following function(s):\n{sigs}"
        out.append({"id": uid("code", r["question_id"]), "type": "pytest", "prompt": prompt, "tests": r["test"]})
        if len(out) == n:
            break
    return out


XLAM_TYPES = {"str": "string", "int": "integer", "float": "number", "bool": "boolean"}


def to_openai_tool(t):
    props, required = {}, []
    for name, p in (t.get("parameters") or {}).items():
        raw = str(p.get("type", "str"))
        base = raw.split(",")[0].strip()
        jtype = XLAM_TYPES.get(base, "array" if base.startswith(("List", "list")) else
                               "object" if base.startswith(("Dict", "dict")) else "string")
        props[name] = {"type": jtype, "description": p.get("description", "")}
        if "default" not in p and "optional" not in raw:
            required.append(name)
    return {"type": "function", "function": {"name": t["name"], "description": t.get("description", ""),
            "parameters": {"type": "object", "properties": props, "required": required}}}


def pool_tool(n):
    ds = load_dataset("Salesforce/xlam-function-calling-60k", split="train").shuffle(seed=SEED)
    out = []
    for r in ds.select(range(min(n, len(ds)))):
        answers = json.loads(r["answers"])
        if not answers:
            continue
        out.append({"id": uid("tool", r["query"]), "type": "tool", "prompt": r["query"],
                    "tools": [to_openai_tool(t) for t in json.loads(r["tools"])],
                    "expected_call": answers[0], "expected_calls": answers})
    return out


pool = []
for name, fn in [("math", pool_math), ("phys", pool_phys), ("code", pool_code), ("tool", pool_tool)]:
    try:
        part = fn(POOL[name])
        print(f"{name}: {len(part)} problems")
        pool += part
    except Exception as e:   # e.g. no access to a gated dataset: that source is skipped
        print(f"{name}: SKIPPED -> {type(e).__name__}: {e}")
random.Random(SEED).shuffle(pool)   # mixed order, so every source is represented if the budget runs out

In [ ]:
def same(a, b):
    return a == b or str(a).strip().lower() == str(b).strip().lower()


def tool_full_match(task, text):
    # Every expected call (name + arguments) must be matched exactly, with no extra calls
    got = R.tool_calls_of({"content": R.strip_thinking(text)})
    expected = task["expected_calls"]
    if len(got) != len(expected):
        return False
    remaining = list(got)
    for e in expected:
        match = next((g for g in remaining if g["name"] == e["name"] and g["arguments"] is not None
                      and all(k in g["arguments"] and same(g["arguments"][k], v) for k, v in e["arguments"].items())),
                     None)
        if match is None:
            return False
        remaining.remove(match)
    return True

## B) Generation (resumable)

In [ ]:
GEN_PATH = f"{RFT_DIR}/generations.jsonl"
done = set()
if os.path.exists(GEN_PATH):
    with open(GEN_PATH, encoding="utf-8") as f:
        done = {json.loads(l)["id"] for l in f if l.strip()}
todo = [t for t in pool if t["id"] not in done]
print(f"Done: {len(done)}, remaining: {len(todo)}")

start, chunk_hours = time.time(), 0.0
for i in range(0, len(todo), CHUNK):
    # Do not start the next chunk if it would exceed the budget (the last chunk's duration is the estimate)
    if (time.time() - start) / 3600 + chunk_hours > MAX_GEN_HOURS:
        print("Time budget reached, stopping.")
        break
    chunk = todo[i:i + CHUNK]
    t0 = time.time()
    samples = generate(chunk, N_SAMPLES, GEN_TEMPERATURE, GEN_MAX_TOKENS)
    with open(GEN_PATH, "a", encoding="utf-8") as f:
        for t, ss in zip(chunk, samples):
            if t["type"] == "tool":
                for s in ss:
                    s["ok"] = s["finish"] == "stop" and tool_full_match(t, s["text"])
            f.write(json.dumps({"id": t["id"], "task": t, "samples": ss}, ensure_ascii=False) + "\n")
    dt = time.time() - t0
    chunk_hours = dt / 3600
    toks = sum(s["tokens"] for ss in samples for s in ss)
    acc = sum(s["ok"] for ss in samples for s in ss) / max(1, sum(len(ss) for ss in samples))
    hours = (time.time() - start) / 3600
    print(f"{i + len(chunk)}/{len(todo)} | {toks / dt:.0f} tok/s | accuracy {acc:.0%} | "
          f"{hours:.2f} h ≈ {hours * CU_PER_HOUR:.1f} CU")

## B) Selection → SFT data

This cell is a preview: it shows the yield per source. `04_sft` re-selects from `generations.jsonl` with its own settings.

- Only **correct and complete** solutions are used, and the **shortest** one is picked. This shortens the reasoning.
- Easy problems (solved 4/4) are kept at the `EASY_KEEP` rate. The real learning signal comes from partially solved problems.
- Problems that were never solved are skipped and listed for a later round.

In [ ]:
def to_example(task, text):
    reasoning, _, answer = text.partition("</think>")
    reasoning, answer = reasoning.strip(), answer.strip()
    msgs = R.build_messages(task)
    if task["type"] == "tool":
        calls = R.tool_calls_of({"content": answer})
        msgs.append({"role": "assistant", "content": answer.split("<tool_call>")[0].strip(),
                     "reasoning_content": reasoning,
                     "tool_calls": [{"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                                    for c in calls]})
        return {"messages": msgs, "tools": task["tools"]}
    msgs.append({"role": "assistant", "content": answer, "reasoning_content": reasoning})
    return {"messages": msgs}


rng = random.Random(SEED)
stats = collections.defaultdict(collections.Counter)
examples, unsolved = [], []
with open(GEN_PATH, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        t, ss = row["task"], row["samples"]
        src = t["id"].split("-")[0]
        good = [s for s in ss if s["ok"]]
        st = stats[src]
        st["problems"] += 1
        st["solved"] += bool(good)
        st["correct_samples"] += len(good)
        st["total_samples"] += len(ss)
        if not good:
            unsolved.append(t["id"])
            continue
        if len(good) == len(ss) and rng.random() > EASY_KEEP:
            continue
        best = min(good, key=lambda s: s["tokens"])
        st["kept"] += 1
        st["picked_tok"] += best["tokens"]
        st["avg_correct_tok"] += sum(s["tokens"] for s in good) / len(good)
        examples.append({**to_example(t, best["text"]), "source": src, "id": t["id"],
                         "pass_rate": len(good) / len(ss), "tokens": best["tokens"]})

random.Random(SEED).shuffle(examples)
with open(f"{RFT_DIR}/rft_sft.jsonl", "w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
with open(f"{RFT_DIR}/unsolved_ids.json", "w") as f:
    json.dump(unsolved, f)

print(f"{'source':<6} {'probs':>6} {'solved':>8} {'pass@1':>7} {'kept':>8} {'picked tok':>12} {'avg correct tok':>15}")
for src, st in stats.items():
    k = max(1, st["kept"])
    print(f"{src:<6} {st['problems']:>6} {st['solved']:>8} {st['correct_samples'] / st['total_samples']:>7.1%} "
          f"{st['kept']:>8} {st['picked_tok'] / k:>12.0f} {st['avg_correct_tok'] / k:>15.0f}")
print(f"\nTotal SFT examples: {len(examples)} -> {RFT_DIR}/rft_sft.jsonl")

In [ ]:
# Check: this is how the SFT notebook will render these examples with the model's own template
for src in ["math", "tool"]:
    ex = next((e for e in examples if e["source"] == src), None)
    if ex:
        print(f"===== {src} =====")
        print(tok.apply_chat_template(ex["messages"], tools=ex.get("tools"), tokenize=False)[-1500:])

## Finish: stop spending CU

In [ ]:
from google.colab import runtime
runtime.unassign()